# Experimentos — Detección de Desviaciones Emocionales

Este notebook es **autocontenido**: genera audio sintético con eventos emocionales
inyectados en posiciones conocidas, corre el pipeline completo y mide si los detectó.

No necesita archivo de audio externo ni token de HuggingFace.

## Diseño del experimento

```
Generamos 2 hablantes sintéticos con F0 y timbre distintos
         ↓
Inyectamos picos emocionales en posiciones conocidas (ground truth)
         ↓
Corremos extracción de features + baseline + detección
         ↓
Comparamos eventos detectados vs. eventos reales → precision / recall
```

In [ ]:
# %pip install librosa praat-parselmouth soundfile pandas matplotlib seaborn scipy numpy

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import parselmouth
from parselmouth.praat import call
from scipy.ndimage import uniform_filter1d
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)   # reproducibilidad
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
OUTPUT = Path('output_experiments')
OUTPUT.mkdir(exist_ok=True)
print('OK')

## 2. Generación de audio sintético

### Modelo de voz simplificado

Simulamos la voz con una **fuente glotal + filtro de tracto vocal**:
- **Fuente**: serie de armónicos a múltiplos de F0 (tono fundamental)
- **Modulación de amplitud**: imita la sílaba (ataque rápido, decaimiento suave)
- **Ruido de canal**: simula compresión telefónica
- **Segmentos de silencio**: entre turnos de habla

Los **eventos emocionales** se inyectan subiendo F0 × `pitch_factor` y la amplitud × `energy_factor` durante un intervalo concreto.

In [ ]:
SR = 16_000   # Hz


def voiced_segment(duration_s: float, f0: float, sr: int,
                   amplitude: float = 0.3,
                   n_harmonics: int = 8,
                   jitter_pct: float = 0.005,
                   noise_level: float = 0.02) -> np.ndarray:
    """
    Genera un segmento de voz sintética:
    - Suma de armónicos de F0 con caída 1/k en amplitud
    - Pequeño jitter en la fase para sonar menos mecánico
    - Modulación de amplitud tipo sílaba
    - Ruido aditivo (canal telefónico)
    """
    n = int(duration_s * sr)
    t = np.arange(n) / sr
    signal = np.zeros(n)

    for k in range(1, n_harmonics + 1):
        freq = f0 * k
        if freq >= sr / 2:
            break
        phase_jitter = rng.uniform(-jitter_pct, jitter_pct) * 2 * np.pi
        signal += (1.0 / k) * np.sin(2 * np.pi * freq * t + phase_jitter)

    # Modulación de amplitud: imitar ritmo silábico (~4 Hz)
    syl_rate = 4.0
    am = 0.5 + 0.5 * np.abs(np.sin(np.pi * syl_rate * t))
    signal = signal * am * amplitude

    # Ruido de canal
    signal += rng.normal(0, noise_level, n)

    return signal.astype(np.float32)


def silence(duration_s: float, sr: int, noise_level: float = 0.003) -> np.ndarray:
    """Silencio con ruido de fondo mínimo."""
    n = int(duration_s * sr)
    return rng.normal(0, noise_level, n).astype(np.float32)


print('Funciones de síntesis definidas.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Perfil de cada hablante
# ─────────────────────────────────────────────────────────────────────────────

SPEAKERS = {
    'SPEAKER_A': dict(
        f0_base    = 130.0,   # Hz — voz masculina grave
        amplitude  = 0.30,
        n_harmonics= 10,
        noise_level= 0.018,
    ),
    'SPEAKER_B': dict(
        f0_base    = 210.0,   # Hz — voz femenina
        amplitude  = 0.28,
        n_harmonics= 8,
        noise_level= 0.020,
    ),
}

# ─────────────────────────────────────────────────────────────────────────────
# Guión de la llamada
# Cada entrada: (hablante, duración_s, factor_pitch, factor_energy, etiqueta)
#   factor_pitch  = 1.0 → neutral, > 1.3 → activación emocional
#   factor_energy = 1.0 → neutral, > 1.5 → más volumen
# ─────────────────────────────────────────────────────────────────────────────

SCRIPT = [
    # (speaker,     dur_s,  f0_factor, amp_factor, label)
    ('SPEAKER_A',    4.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    5.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_A',    3.5,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    4.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_A',    6.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    5.5,    1.00,      1.00,   'neutro'),
    ('SPEAKER_A',    4.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    3.0,    1.00,      1.00,   'neutro'),

    # ── Evento A: SPEAKER_A se molesta (~t=40s) ───────────────
    ('SPEAKER_A',    7.0,    1.45,      1.80,   'EVENTO_ENOJO_A'),
    ('SPEAKER_B',    3.5,    1.10,      1.20,   'neutro_reactivo'),   # B reacciona levemente
    ('SPEAKER_A',    5.0,    1.30,      1.50,   'EVENTO_ENOJO_A_cont'),

    # ── Vuelta a la normalidad ────────────────────────────────
    ('SPEAKER_A',    4.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    6.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_A',    5.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    4.5,    1.00,      1.00,   'neutro'),
    ('SPEAKER_A',    3.0,    1.00,      1.00,   'neutro'),

    # ── Evento B: SPEAKER_B se emociona/exalta (~t=80s) ──────
    ('SPEAKER_B',    8.0,    1.50,      1.70,   'EVENTO_EXCITACION_B'),
    ('SPEAKER_A',    4.0,    1.05,      1.10,   'neutro_reactivo'),
    ('SPEAKER_B',    5.0,    1.35,      1.40,   'EVENTO_EXCITACION_B_cont'),

    # ── Cierre tranquilo ──────────────────────────────────────
    ('SPEAKER_A',    5.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    4.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_A',    6.0,    1.00,      1.00,   'neutro'),
    ('SPEAKER_B',    3.5,    1.00,      1.00,   'neutro'),
]

# ─────────────────────────────────────────────────────────────────────────────
# Construir audio y tabla de segmentos (ground truth)
# ─────────────────────────────────────────────────────────────────────────────

audio_chunks = []
gt_rows = []          # ground truth de segmentos
t_cursor = 0.0

for spk, dur, f0_f, amp_f, label in SCRIPT:
    p = SPEAKERS[spk]
    chunk = voiced_segment(
        duration_s  = dur,
        f0          = p['f0_base'] * f0_f,
        sr          = SR,
        amplitude   = p['amplitude'] * amp_f,
        n_harmonics = p['n_harmonics'],
        noise_level = p['noise_level'],
    )
    # Pequeño silencio entre turnos
    sil = silence(rng.uniform(0.1, 0.4), SR)

    audio_chunks.append(chunk)
    audio_chunks.append(sil)

    gt_rows.append(dict(
        speaker  = spk,
        start    = t_cursor,
        end      = t_cursor + dur,
        duration = dur,
        label    = label,
        is_event = 'EVENTO' in label,
    ))
    t_cursor += dur + len(sil) / SR

y_full = np.concatenate(audio_chunks)
# Normalizar a ±0.9
y_full = y_full / (np.abs(y_full).max() + 1e-9) * 0.9

df_gt = pd.DataFrame(gt_rows)
# Simulamos «diarización perfecta» con el ground truth
df_segments = df_gt[['speaker','start','end','duration']].copy()

total_s = len(y_full) / SR
print(f'Audio sintético generado: {total_s:.1f} s ({total_s/60:.2f} min)')
print(f'Segmentos: {len(df_gt)}')
print('\nEventos inyectados (ground truth):')
display(df_gt[df_gt['is_event']][['speaker','start','end','label']].round(1))

# Guardar WAV para escuchar / usar en pipeline real
wav_path = OUTPUT / 'llamada_sintetica.wav'
sf.write(str(wav_path), y_full, SR)
print(f'\nAudio guardado en: {wav_path}')

## 3. Visualización del audio generado

In [ ]:
SPK_COLORS = {'SPEAKER_A': '#2196F3', 'SPEAKER_B': '#FF5722'}
EVENT_COLOR = '#E91E63'


def plot_waveform_gt(y, sr, df_gt):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 6),
                                    gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    t = np.linspace(0, len(y) / sr, len(y))

    # Forma de onda con color por hablante
    ax1.plot(t, y, color='#CCCCCC', linewidth=0.3, zorder=1)
    for _, row in df_gt.iterrows():
        mask = (t >= row['start']) & (t <= row['end'])
        ax1.fill_between(t, y, where=mask,
                         color=SPK_COLORS[row['speaker']], alpha=0.45, zorder=2)

    # Zonas de evento (ground truth) como rectángulos
    for _, row in df_gt[df_gt['is_event']].iterrows():
        ax1.axvspan(row['start'], row['end'], color=EVENT_COLOR, alpha=0.20, zorder=3)
        ax1.annotate(row['label'].replace('_', '\n'),
                     xy=((row['start'] + row['end']) / 2,
                         y.max() * 0.85),
                     fontsize=7.5, ha='center', color=EVENT_COLOR, fontweight='bold')

    patches = [mpatches.Patch(color=SPK_COLORS[s], label=s)
               for s in SPK_COLORS]
    patches.append(mpatches.Patch(color=EVENT_COLOR, alpha=0.5, label='Evento inyectado'))
    ax1.legend(handles=patches, fontsize=9)
    ax1.set_ylabel('Amplitud')
    ax1.set_title('Audio sintético — forma de onda (ground truth)', fontsize=12)

    # Gantt de turnos
    for spk_i, spk in enumerate(SPK_COLORS):
        segs = df_gt[df_gt['speaker'] == spk]
        for _, row in segs.iterrows():
            color = EVENT_COLOR if row['is_event'] else SPK_COLORS[spk]
            ax2.barh(spk_i, row['duration'], left=row['start'],
                     color=color, height=0.5, alpha=0.85)

    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(list(SPK_COLORS.keys()))
    ax2.set_xlabel('Tiempo (s)')
    ax2.set_title('Turnos de habla (diarización simulada)')

    plt.tight_layout()
    plt.savefig(OUTPUT / 'waveform_gt.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_waveform_gt(y_full, SR, df_gt)

## 4. Escuchar fragmentos del audio

Comparamos un segmento neutro vs. el evento emocional de cada hablante.

In [ ]:
from IPython.display import Audio, display as ipy_display


def play_segment(label_substr: str, df_gt=df_gt, y=y_full, sr=SR, margin=1.0):
    row = df_gt[df_gt['label'].str.contains(label_substr)].iloc[0]
    s = max(0, int((row['start'] - margin) * sr))
    e = min(len(y), int((row['end']   + margin) * sr))
    print(f"▶ {row['label']}  [{row['start']:.1f}s – {row['end']:.1f}s]")
    return Audio(y[s:e], rate=sr)


print('=== SPEAKER_A — neutro ===')
ipy_display(play_segment('neutro'))

print('\n=== SPEAKER_A — evento enojo ===')
ipy_display(play_segment('EVENTO_ENOJO_A'))

print('\n=== SPEAKER_B — neutro ===')
ipy_display(play_segment('neutro'))

print('\n=== SPEAKER_B — evento excitación ===')
ipy_display(play_segment('EVENTO_EXCITACION_B'))

## 5. Extracción de features en ventanas deslizantes

In [ ]:
WINDOW_S = 4.0
HOP_S    = 0.5    # hop corto para buena resolución temporal

FEATURE_WEIGHTS = {
    'pitch_mean':  1.5,
    'pitch_std':   1.2,
    'energy_db':   1.3,
    'speech_rate': 1.0,
    'jitter':      0.8,
    'hnr':         0.7,
}
FEAT_COLS = list(FEATURE_WEIGHTS.keys())


def dominant_speaker(t0, t1, df_seg):
    ov = df_seg.copy()
    ov['ov'] = ov.apply(lambda r: max(0, min(r['end'], t1) - max(r['start'], t0)), axis=1)
    best = ov.loc[ov['ov'].idxmax()]
    return best['speaker'] if best['ov'] > 0 else 'SILENCE'


def features_for_window(y_win, sr):
    f = {}

    # Pitch
    f0, voiced, _ = librosa.pyin(y_win, fmin=75, fmax=500, sr=sr)
    vf0 = f0[voiced]
    f['pitch_mean'] = float(np.mean(vf0))  if len(vf0) >= 4 else np.nan
    f['pitch_std']  = float(np.std(vf0))   if len(vf0) >= 4 else np.nan

    # Energía
    rms = librosa.feature.rms(y=y_win)[0]
    f['energy_db'] = float(librosa.amplitude_to_db(rms, ref=1.0).mean())

    # Tasa de habla (onsets)
    onsets = librosa.onset.onset_detect(y=y_win, sr=sr, units='time')
    f['speech_rate'] = len(onsets) / (len(y_win) / sr)

    # Jitter y HNR (Praat)
    try:
        snd = parselmouth.Sound(y_win.astype(np.float64), sampling_frequency=sr)
        pp  = call(snd, 'To PointProcess (periodic, cc)', 75, 500)
        j   = call(pp,  'Get jitter (local)', 0, 0, 0.0001, 0.02, 1.3)
        f['jitter'] = float(j) if np.isfinite(j) else np.nan
        h   = call(snd, 'To Harmonicity (cc)', 0.01, 75, 0.1, 1.0)
        hnr = call(h,   'Get mean', 0, 0)
        f['hnr'] = float(hnr) if np.isfinite(hnr) else np.nan
    except Exception:
        f['jitter'] = np.nan
        f['hnr']    = np.nan

    return f


def extract_windows(y, sr, df_seg, window_s, hop_s):
    total_s = len(y) / sr
    rows = []
    t = 0.0
    n_wins = int((total_s - window_s) / hop_s) + 1

    for i in range(n_wins):
        t1 = t + window_s
        spk = dominant_speaker(t, t1, df_seg)
        if spk != 'SILENCE':
            y_win = y[int(t * sr): int(t1 * sr)]
            f = features_for_window(y_win, sr)
            f['time_s']  = t
            f['speaker'] = spk
            rows.append(f)
        t += hop_s
        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{n_wins}...')

    return pd.DataFrame(rows)


print(f'Extrayendo features ({WINDOW_S}s / hop {HOP_S}s)...')
df_wins = extract_windows(y_full, SR, df_segments, WINDOW_S, HOP_S)
print(f'Listo: {len(df_wins)} ventanas')
df_wins.head()

## 6. Línea base y score de desviación

In [ ]:
BASELINE_S = 30.0   # primeros N segundos de habla = base neutra


def compute_baseline(df_wins, speaker, baseline_s, hop_s):
    df_spk = df_wins[df_wins['speaker'] == speaker].sort_values('time_s')
    k = max(5, int(baseline_s / hop_s))
    df_base = df_spk.head(k)
    base = {}
    for col in FEAT_COLS:
        v = df_base[col].dropna()
        base[col] = {'mean': v.mean(), 'std': max(v.std(), 1e-6)}
    return base


def add_scores(df_wins, baselines):
    df = df_wins.copy()
    for spk, base in baselines.items():
        mask = df['speaker'] == spk
        for feat, w in FEATURE_WEIGHTS.items():
            mu, sd = base[feat]['mean'], base[feat]['std']
            df.loc[mask, f'z_{feat}'] = (df.loc[mask, feat] - mu) / sd

        z_cols = [f'z_{f}' for f in FEATURE_WEIGHTS]
        w_arr  = np.array(list(FEATURE_WEIGHTS.values()))
        z_mat  = df.loc[mask, z_cols].values
        df.loc[mask, 'emotion_score'] = np.nansum(np.abs(z_mat) * w_arr, axis=1)

        dir_sig = df.loc[mask, 'z_pitch_mean'].fillna(0) + \
                  df.loc[mask, 'z_energy_db'].fillna(0)
        df.loc[mask, 'direction'] = np.sign(dir_sig)

    # Suavizado
    for spk in baselines:
        mask = df['speaker'] == spk
        df.loc[mask, 'score_smooth'] = uniform_filter1d(
            df.loc[mask, 'emotion_score'].fillna(0), size=7
        )
    return df


baselines = {spk: compute_baseline(df_wins, spk, BASELINE_S, HOP_S)
             for spk in SPK_COLORS}
df_wins = add_scores(df_wins, baselines)

print('Score calculado. Resumen por hablante:')
print(df_wins.groupby('speaker')['score_smooth']
      .agg(['mean','std','max']).round(3))

## 7. Detección de eventos

In [ ]:
ALERT_THRESHOLD_SD = 1.8   # desviaciones estándar sobre la media del score
MIN_EVENT_DURATION = 2.0   # ignorar eventos de menos de N segundos


def detect_events(df_wins, threshold_sd, min_dur, hop_s):
    df = df_wins.copy()
    events = []

    for spk in df['speaker'].unique():
        mask   = df['speaker'] == spk
        df_spk = df[mask].sort_values('time_s')

        mu  = df_spk['score_smooth'].mean()
        sd  = df_spk['score_smooth'].std()
        thr = mu + threshold_sd * sd
        df.loc[mask, 'alert']     = df_spk['score_smooth'] > thr
        df.loc[mask, 'threshold'] = thr

        alert_df = df_spk[df_spk['score_smooth'] > thr].copy()
        if alert_df.empty:
            continue

        alert_df['gap']   = alert_df['time_s'].diff().gt(2 * hop_s)
        alert_df['group'] = alert_df['gap'].cumsum()

        for _, grp in alert_df.groupby('group'):
            dur = grp['time_s'].max() - grp['time_s'].min() + WINDOW_S
            if dur < min_dur:
                continue
            events.append(dict(
                speaker        = spk,
                start_s        = grp['time_s'].min(),
                end_s          = grp['time_s'].max() + WINDOW_S,
                duration_s     = dur,
                peak_score     = grp['score_smooth'].max(),
                mean_direction = grp['direction'].mean(),
            ))

    df_events = pd.DataFrame(events)
    if not df_events.empty:
        df_events['tipo'] = df_events['mean_direction'].map(
            lambda d: 'Alta activación' if d > 0 else 'Baja activación'
        )
    return df, df_events


df_wins, df_events = detect_events(df_wins, ALERT_THRESHOLD_SD, MIN_EVENT_DURATION, HOP_S)

print('\n=== Eventos detectados ===')
if not df_events.empty:
    display(df_events.round(2))
else:
    print('Ninguno. Ajusta ALERT_THRESHOLD_SD.')

## 8. Evaluación: detectados vs. ground truth

Como inyectamos los eventos en posiciones conocidas, podemos medir **precisión y recall**
usando solapamiento temporal (IoU).

In [ ]:
def iou(a_start, a_end, b_start, b_end):
    inter = max(0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return inter / union if union > 0 else 0.0


def evaluate(df_gt, df_events, iou_threshold=0.3):
    # Agrupar segmentos de evento GT contiguos del mismo hablante
    gt_events = []
    for spk in df_gt['speaker'].unique():
        df_ev = df_gt[(df_gt['speaker'] == spk) & df_gt['is_event']].copy()
        if df_ev.empty:
            continue
        df_ev = df_ev.sort_values('start')
        # Fusionar segmentos contiguos (gap < 1s)
        cur_start, cur_end, cur_label = df_ev.iloc[0]['start'], df_ev.iloc[0]['end'], df_ev.iloc[0]['label']
        for _, row in df_ev.iloc[1:].iterrows():
            if row['start'] - cur_end < 1.0:
                cur_end   = row['end']
                cur_label = row['label']
            else:
                gt_events.append(dict(speaker=spk, start=cur_start, end=cur_end, label=cur_label))
                cur_start, cur_end, cur_label = row['start'], row['end'], row['label']
        gt_events.append(dict(speaker=spk, start=cur_start, end=cur_end, label=cur_label))

    df_gt_ev = pd.DataFrame(gt_events)

    if df_events.empty:
        print('Sin detecciones — recall = 0')
        return df_gt_ev, pd.DataFrame()

    # Matching greedy por IoU
    matched_gt  = set()
    matched_det = set()
    matches = []

    for di, det in df_events.iterrows():
        best_iou, best_gi = 0.0, None
        for gi, gt in df_gt_ev.iterrows():
            if gt['speaker'] != det['speaker']:
                continue
            score = iou(det['start_s'], det['end_s'], gt['start'], gt['end'])
            if score > best_iou:
                best_iou, best_gi = score, gi
        if best_iou >= iou_threshold and best_gi not in matched_gt:
            matched_gt.add(best_gi)
            matched_det.add(di)
            matches.append(dict(
                speaker     = det['speaker'],
                gt_label    = df_gt_ev.loc[best_gi, 'label'],
                gt_start    = df_gt_ev.loc[best_gi, 'start'],
                gt_end      = df_gt_ev.loc[best_gi, 'end'],
                det_start   = det['start_s'],
                det_end     = det['end_s'],
                iou         = round(best_iou, 3),
            ))

    TP        = len(matched_det)
    FP        = len(df_events) - TP
    FN        = len(df_gt_ev)  - TP
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f'\nGround truth: {len(df_gt_ev)} eventos')
    print(f'Detectados  : {len(df_events)} eventos')
    print(f'\n  TP = {TP}  FP = {FP}  FN = {FN}')
    print(f'  Precision = {precision:.2f}')
    print(f'  Recall    = {recall:.2f}')
    print(f'  F1        = {f1:.2f}')

    if matches:
        print('\nCoincidencias:')
        display(pd.DataFrame(matches).round(2))

    return df_gt_ev, pd.DataFrame(matches)


df_gt_events, df_matches = evaluate(df_gt, df_events, iou_threshold=0.3)

## 9. Arco emocional completo

In [ ]:
def plot_emotional_arc(y, sr, df_wins, df_events, df_gt_events):
    speakers = list(SPK_COLORS.keys())
    fig = plt.figure(figsize=(20, 4 + 5 * len(speakers)))
    gs  = gridspec.GridSpec(1 + len(speakers), 1, figure=fig, hspace=0.5)

    # ── Forma de onda global ───────────────────────────────────
    ax0 = fig.add_subplot(gs[0])
    t   = np.linspace(0, len(y) / sr, len(y))
    ax0.plot(t, y, color='#DDDDDD', linewidth=0.25, zorder=1)
    for _, row in df_segments.iterrows():
        mask = (t >= row['start']) & (t <= row['end'])
        ax0.fill_between(t, y, where=mask,
                         color=SPK_COLORS[row['speaker']], alpha=0.35, zorder=2)
    for _, ev in df_gt_events.iterrows():
        ax0.axvspan(ev['start'], ev['end'], color=EVENT_COLOR, alpha=0.18, zorder=3,
                    label='GT evento')
    for _, ev in df_events.iterrows():
        ax0.axvspan(ev['start_s'], ev['end_s'], color='black', alpha=0.08,
                    linestyle='--', zorder=4, label='Detectado')

    h = [mpatches.Patch(color=SPK_COLORS[s], label=s) for s in speakers]
    h += [mpatches.Patch(color=EVENT_COLOR, alpha=0.5, label='GT evento'),
          mpatches.Patch(color='black',     alpha=0.3, label='Detectado')]
    ax0.legend(handles=h, fontsize=9, ncol=4)
    ax0.set_title('Forma de onda — ground truth (rosa) vs. detecciones (gris)', fontsize=11)
    ax0.set_ylabel('Amplitud')

    # ── Score emocional por hablante ───────────────────────────
    for i, spk in enumerate(speakers):
        ax = fig.add_subplot(gs[i + 1])
        df_s = df_wins[df_wins['speaker'] == spk].sort_values('time_s')

        t_s  = df_s['time_s'].values
        sc   = df_s['score_smooth'].values
        thr  = df_s['threshold'].iloc[0] if 'threshold' in df_s.columns else np.nan
        ddir = df_s['direction'].values

        # Área bajo el score coloreada por dirección
        ax.fill_between(t_s, sc, where=(ddir > 0),
                        color='#FF5722', alpha=0.25, label='Alta activación')
        ax.fill_between(t_s, sc, where=(ddir <= 0),
                        color='#607D8B', alpha=0.20, label='Baja activación')
        ax.plot(t_s, sc, color=SPK_COLORS[spk], linewidth=1.8, zorder=3)

        # Líneas de referencia
        ax.axhline(df_s['score_smooth'].mean(), color='green',
                   linestyle='--', linewidth=1.2, alpha=0.7, label='Media base')
        if np.isfinite(thr):
            ax.axhline(thr, color='red', linestyle=':', linewidth=1.8,
                       label=f'Umbral ({ALERT_THRESHOLD_SD}σ)')

        # Sombrear eventos GT
        gt_spk = df_gt_events[df_gt_events['speaker'] == spk] if not df_gt_events.empty else pd.DataFrame()
        for _, gev in gt_spk.iterrows():
            ax.axvspan(gev['start'], gev['end'],
                       color=EVENT_COLOR, alpha=0.15, zorder=1, label='GT')

        # Marcar detecciones
        det_spk = df_events[df_events['speaker'] == spk] if not df_events.empty else pd.DataFrame()
        for _, dev in det_spk.iterrows():
            ax.axvspan(dev['start_s'], dev['end_s'],
                       color='black', alpha=0.07, zorder=2)
            ax.annotate('⚡', xy=((dev['start_s'] + dev['end_s']) / 2,
                                   thr * 1.02 if np.isfinite(thr) else sc.max()),
                        fontsize=15, ha='center', color='red')

        ax.set_title(f'Score emocional — {spk}', fontsize=11)
        ax.set_ylabel('Desviación de base')
        ax.set_xlabel('Tiempo (s)')
        ax.legend(fontsize=8, ncol=3, loc='upper right')

    plt.suptitle('Arco Emocional — Experimento con audio sintético',
                 fontsize=14, y=1.01, fontweight='bold')
    plt.savefig(OUTPUT / 'arc_emotional.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_emotional_arc(y_full, SR, df_wins, df_events, df_gt_events)

## 10. Heatmap de z-scores — qué features dispararon el evento

In [ ]:
def plot_zscore_heatmap(df_wins, df_events, df_gt_events):
    z_cols  = [f'z_{f}' for f in FEATURE_WEIGHTS if f'z_{f}' in df_wins.columns]
    labels  = list(FEATURE_WEIGHTS.keys())
    cmap    = LinearSegmentedColormap.from_list('emo', ['#1565C0', '#FAFAFA', '#C62828'])

    for spk in SPK_COLORS:
        df_s = df_wins[df_wins['speaker'] == spk].sort_values('time_s')
        if df_s.empty:
            continue

        z_mat  = np.clip(df_s[z_cols].values.T, -4, 4)
        t_vals = df_s['time_s'].values

        fig, axes = plt.subplots(2, 1, figsize=(18, 6),
                                  gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

        # Heatmap
        im = axes[0].imshow(
            z_mat, aspect='auto', cmap=cmap, vmin=-4, vmax=4,
            extent=[t_vals[0], t_vals[-1], len(z_cols) - 0.5, -0.5]
        )
        axes[0].set_yticks(range(len(labels)))
        axes[0].set_yticklabels(labels, fontsize=10)
        axes[0].set_title(f'Z-scores por feature — {spk}  '
                           '(rojo = alta, azul = baja activación)', fontsize=11)
        plt.colorbar(im, ax=axes[0], label='z-score', shrink=0.8)

        # Sombrear eventos GT y detecciones
        gt_spk = df_gt_events[df_gt_events['speaker'] == spk] if not df_gt_events.empty else pd.DataFrame()
        det_spk = df_events[df_events['speaker'] == spk] if not df_events.empty else pd.DataFrame()
        for _, r in gt_spk.iterrows():
            axes[0].axvspan(r['start'], r['end'], color=EVENT_COLOR, alpha=0.18, zorder=5)
        for _, r in det_spk.iterrows():
            axes[0].axvspan(r['start_s'], r['end_s'], color='black', alpha=0.10, zorder=6)

        # Score en la parte inferior
        t_s  = df_s['time_s'].values
        sc   = df_s['score_smooth'].values
        thr  = df_s['threshold'].iloc[0] if 'threshold' in df_s.columns else np.nan
        axes[1].plot(t_s, sc, color=SPK_COLORS[spk], linewidth=1.5)
        axes[1].fill_between(t_s, sc, alpha=0.3, color=SPK_COLORS[spk])
        if np.isfinite(thr):
            axes[1].axhline(thr, color='red', linestyle=':', linewidth=1.5)
        for _, r in gt_spk.iterrows():
            axes[1].axvspan(r['start'], r['end'], color=EVENT_COLOR, alpha=0.18)
        axes[1].set_ylabel('Score')
        axes[1].set_xlabel('Tiempo (s)')

        plt.tight_layout()
        plt.savefig(OUTPUT / f'heatmap_{spk}.png', dpi=150, bbox_inches='tight')
        plt.show()


plot_zscore_heatmap(df_wins, df_events, df_gt_events)

## 11. Distribución de features: neutro vs. evento

In [ ]:
def plot_feature_distributions(df_wins, df_gt):
    """
    KDE de cada feature separando ventanas neutras vs. ventanas dentro de un evento GT.
    Muestra directamente cuánto cambia cada feature durante la emoción.
    """
    def is_in_event(t_s, spk, df_gt):
        ev = df_gt[(df_gt['speaker'] == spk) & df_gt['is_event']]
        return any((t_s >= r['start']) & (t_s <= r['end']) for _, r in ev.iterrows())

    df_w = df_wins.copy()
    df_w['in_event'] = df_w.apply(
        lambda r: is_in_event(r['time_s'], r['speaker'], df_gt), axis=1
    )
    df_w['label'] = df_w['in_event'].map({True: 'Evento', False: 'Neutro'})

    plot_feats = ['pitch_mean', 'pitch_std', 'energy_db', 'speech_rate', 'jitter', 'hnr']
    feat_labels = ['Pitch medio (Hz)', 'Pitch std (Hz)', 'Energía (dB)',
                   'Tasa habla (onsets/s)', 'Jitter', 'HNR (dB)']

    for spk in SPK_COLORS:
        df_s = df_w[df_w['speaker'] == spk]
        if df_s.empty:
            continue

        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        axes = axes.flatten()

        for ax, feat, lab in zip(axes, plot_feats, feat_labels):
            d_neutro = df_s[df_s['in_event'] == False][feat].dropna()
            d_evento = df_s[df_s['in_event'] == True][feat].dropna()

            if len(d_neutro) > 2:
                d_neutro.plot.kde(ax=ax, color='steelblue', linewidth=2,
                                  label=f'Neutro (μ={d_neutro.mean():.1f})')
                ax.axvline(d_neutro.mean(), color='steelblue', linestyle='--', alpha=0.6)

            if len(d_evento) > 2:
                d_evento.plot.kde(ax=ax, color=EVENT_COLOR, linewidth=2,
                                  label=f'Evento (μ={d_evento.mean():.1f})')
                ax.axvline(d_evento.mean(), color=EVENT_COLOR, linestyle='--', alpha=0.6)

            ax.set_title(lab, fontsize=10)
            ax.set_xlabel('')
            ax.legend(fontsize=8)

        fig.suptitle(f'Distribución de features — {spk}  '
                     '(neutro vs. durante evento emocional)',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUTPUT / f'distributions_{spk}.png', dpi=150, bbox_inches='tight')
        plt.show()


plot_feature_distributions(df_wins, df_gt)

## 12. Experimento de sensibilidad: ¿cómo cambia el F1 con el umbral?

Barremos distintos valores de `ALERT_THRESHOLD_SD` y graficamos precision/recall.

In [ ]:
def sweep_thresholds(df_wins, df_gt, df_gt_events, thresholds):
    results = []
    for thr in thresholds:
        _, df_ev = detect_events(df_wins, thr, MIN_EVENT_DURATION, HOP_S)
        if df_ev.empty:
            results.append(dict(threshold=thr, precision=0, recall=0, f1=0, n_det=0))
            continue

        # Contar matches
        matched = 0
        for _, det in df_ev.iterrows():
            for _, gt in df_gt_events[df_gt_events['speaker'] == det['speaker']].iterrows():
                if iou(det['start_s'], det['end_s'], gt['start'], gt['end']) >= 0.3:
                    matched += 1
                    break

        tp = matched
        fp = len(df_ev) - tp
        fn = len(df_gt_events) - tp
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
        results.append(dict(threshold=thr, precision=p, recall=r, f1=f1, n_det=len(df_ev)))

    return pd.DataFrame(results)


thresholds = np.arange(0.8, 3.5, 0.2)
print('Evaluando umbrales...')
df_sweep = sweep_thresholds(df_wins, df_gt, df_gt_events, thresholds)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(df_sweep['threshold'], df_sweep['precision'],
         'o-', color='steelblue', label='Precision', linewidth=2)
ax1.plot(df_sweep['threshold'], df_sweep['recall'],
         's-', color='darkorange', label='Recall', linewidth=2)
ax1.plot(df_sweep['threshold'], df_sweep['f1'],
         '^-', color='green', label='F1', linewidth=2.5)
ax1.axvline(ALERT_THRESHOLD_SD, color='red', linestyle='--',
            alpha=0.7, label=f'Umbral actual ({ALERT_THRESHOLD_SD})')
ax1.set_xlabel('ALERT_THRESHOLD_SD')
ax1.set_ylabel('Métrica')
ax1.set_ylim(0, 1.05)
ax1.set_title('Precision / Recall / F1 vs. umbral')
ax1.legend()

ax2.plot(df_sweep['recall'], df_sweep['precision'],
         'o-', color='purple', linewidth=2, markersize=6)
for _, row in df_sweep.iterrows():
    ax2.annotate(f'{row["threshold"]:.1f}',
                 (row['recall'], row['precision']),
                 fontsize=7, ha='center', va='bottom')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_xlim(0, 1.05)
ax2.set_ylim(0, 1.05)
ax2.set_title('Curva Precision-Recall')

plt.tight_layout()
plt.savefig(OUTPUT / 'threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMejor F1:')
display(df_sweep.loc[df_sweep['f1'].idxmax()].to_frame().T.round(3))

## 13. Experimento: ¿cuánto audio base es necesario?

Probamos con diferentes duraciones de línea base para ver el impacto en la detección.

In [ ]:
baseline_durations = [10, 20, 30, 45, 60]
baseline_results = []

for bd in baseline_durations:
    bl = {spk: compute_baseline(df_wins, spk, bd, HOP_S) for spk in SPK_COLORS}
    dw = add_scores(df_wins.drop(columns=[c for c in df_wins.columns
                                           if c.startswith('z_') or
                                           c in ('emotion_score', 'score_smooth',
                                                  'direction', 'alert', 'threshold')],
                                  errors='ignore'), bl)
    _, dw = detect_events(dw, ALERT_THRESHOLD_SD, MIN_EVENT_DURATION, HOP_S)

    matched = 0
    for _, det in dw.iterrows():
        for _, gt in df_gt_events[df_gt_events['speaker'] == det['speaker']].iterrows():
            if iou(det['start_s'], det['end_s'], gt['start'], gt['end']) >= 0.3:
                matched += 1
                break
    tp = matched
    fp = len(dw) - tp
    fn = len(df_gt_events) - tp
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
    baseline_results.append(dict(baseline_s=bd, precision=p, recall=r, f1=f1))

df_bl = pd.DataFrame(baseline_results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_bl['baseline_s'], df_bl['f1'], 'o-', color='green', linewidth=2.5,
        markersize=8, label='F1')
ax.plot(df_bl['baseline_s'], df_bl['precision'], 's--', color='steelblue',
        linewidth=1.8, label='Precision')
ax.plot(df_bl['baseline_s'], df_bl['recall'], '^--', color='darkorange',
        linewidth=1.8, label='Recall')
ax.axvline(BASELINE_S, color='red', linestyle=':', alpha=0.7,
           label=f'Base actual ({BASELINE_S}s)')
ax.set_xlabel('Duración de la línea base (s)')
ax.set_ylabel('Métrica')
ax.set_ylim(0, 1.05)
ax.set_title('Impacto de la duración de la base en la detección')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'baseline_duration_impact.png', dpi=150, bbox_inches='tight')
plt.show()

display(df_bl.round(3))

## 14. Resumen y conclusiones del experimento

In [ ]:
print('=' * 65)
print('RESUMEN DEL EXPERIMENTO')
print('=' * 65)
print(f'  Audio sintético    : {total_s:.1f} s  ({total_s/60:.1f} min)')
print(f'  Hablantes          : {len(SPK_COLORS)}')
print(f'  Eventos inyectados : {len(df_gt_events)}')
print(f'  Ventana análisis   : {WINDOW_S}s  /  hop {HOP_S}s')
print(f'  Línea base         : primeros {BASELINE_S}s de habla')
print(f'  Umbral detección   : {ALERT_THRESHOLD_SD}σ')
print()

best = df_sweep.loc[df_sweep['f1'].idxmax()]
print(f'  Umbral óptimo      : {best["threshold"]:.1f}σ  →  F1={best["f1"]:.2f}')
print()

print('  Archivos generados en:', OUTPUT)
for f in sorted(OUTPUT.glob('*.png')):
    print(f'    {f.name}')

# Guardar resultados
df_wins.to_csv(OUTPUT / 'windows_full.csv', index=False)
df_events.to_csv(OUTPUT / 'events_detected.csv', index=False)
df_gt_events.to_csv(OUTPUT / 'events_ground_truth.csv', index=False)
df_sweep.to_csv(OUTPUT / 'threshold_sweep.csv', index=False)
print('\nCSVs exportados.')